In [1]:
import torch
import numpy as np
import pandas as pd
from haversine import haversine, Unit
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder, StandardScaler
from graphrfi_subgraphs import *


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

# 1. Load Dataset

In [3]:
trainpath = f'../../../data/top30groups/LongLatCombined/train1/train{partition}.csv'
testpath = f'../../../data/top30groups/LongLatCombined/test1/test{partition}.csv'
traindata = pd.read_csv(trainpath, encoding='ISO-8859-1')
testdata = pd.read_csv(testpath, encoding='ISO-8859-1')

In [4]:
traindata_list, testdata_list, y_gcn, y_nrf, nrf_input, index_to_label = build_graph_data(traindata, testdata, 'weaptype1')

Feature Matrix shape:  (1790, 2)


In [5]:
from itertools import product

param_grid = {
    'embed_dim': [16, 32],
    'lr': [0.001],
    'n_tree': [40, 80],
    'tree_depth': [10],
    'feat_dropout': [0, 0.1],
    'tree_feature_rate': [0.3, 0.5],
    'batch_size': [256]
}

In [6]:
import copy

best_acc = -1
best_params = None
best_epoch = -1
results = []

# Create all combinations of the parameter grid
keys, values = zip(*param_grid.items())
i = 1
total_combinations = len(list(product(*values)))

for v in product(*values):
    # Build argument dict
    print(f"{i}/{total_combinations}")
    params = dict(zip(keys, v))
    
    # Merge with fixed defaults
    args = {
        'partition': f"gtd{partition}",
        'epochs': 1500,
        'n_class': 30,
        'final_evaluation': False,
        **params  # override with params from grid
    }

    print(f"\nRunning: {args}")

    try:
        acc, epoch, *_ = train_joint_subgraph(
            traindata_list,
            testdata_list,
            y_gcn,
            y_nrf,
            nrf_input,
            args,
            index_to_label,
            verbose=False
        )

        results.append((acc, copy.deepcopy(args)))

        if acc > best_acc:
            best_acc = acc
            best_epoch = epoch
            best_params = copy.deepcopy(args)

    except Exception as e:
        print(f"Error with params {params}: {e}")

    i = i + 1

print("\nBest Accuracy:", best_acc)
print("\nBest Epoch:", best_epoch)
print("Best Parameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")


1/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'final_evaluation': False, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0, 'tree_feature_rate': 0.3, 'batch_size': 256}
Best acc/epoch: 0.7571 at epoch 1060
2/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'final_evaluation': False, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0, 'tree_feature_rate': 0.5, 'batch_size': 256}
Best acc/epoch: 0.7881 at epoch 1392
3/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'final_evaluation': False, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0.1, 'tree_feature_rate': 0.3, 'batch_size': 256}
Best acc/epoch: 0.7667 at epoch 1384
4/16

Running: {'partition': 'gtd100', 'epochs': 1500, 'n_class': 30, 'final_evaluation': False, 'embed_dim': 16, 'lr': 0.001, 'n_tree': 40, 'tree_depth': 10, 'feat_dropout': 0.1, 'tree_feature_rate': 0.5, 'batch_size'

In [ ]:
"""partition: gtd100
epochs: 1500
n_class: 30
embed_dim: 32
lr: 0.001
n_tree: 80
tree_depth: 10
feat_dropout: 0
tree_feature_rate: 0.3
batch_size: 256
0.8355555534362793"""

args = {
    'partition': f"gtd{partition}",
    'epochs': 3000,
    'n_class': 30,
    'final_evaluation': True,
    'lr': 0.001,
    'embed_dim': 32,
    'n_tree': 80,
    'tree_depth': 10,
    'feat_dropout': 0,
    'tree_feature_rate': 0.3,
    'batch_size': 256
}

best_acc,best_epoch,precision, recall, f1,y_pred_decoded, y_true_decoded,precision_micro, recall_micro, f1_micro,precision_macro, recall_macro, f1_macro,roc_auc_weighted, roc_auc_micro, roc_auc_macro,epoch_logs = train_joint_subgraph(
        traindata_list,
        testdata_list,
        y_gcn,
        y_nrf,
        nrf_input,
        args,
        index_to_label,
        verbose=True
    )

print(best_acc, best_epoch)

metrics = {
    "best_acc": [best_acc],
    "best_epoch": [best_epoch],
    "precision_weighted": [precision],
    "recall_weighted": [recall],
    "f1_weighted": [f1],
    "precision_micro": [precision_micro],
    "recall_micro": [recall_micro],
    "f1_micro": [f1_micro],
    "precision_macro": [precision_macro],
    "recall_macro": [recall_macro],
    "f1_macro": [f1_macro],
    "roc_auc_weighted": [roc_auc_weighted],
    "roc_auc_micro": [roc_auc_micro],
    "roc_auc_macro": [roc_auc_macro],
}

pd.DataFrame(metrics).to_csv("results.csv", index=False)

Epoch 01 | Joint Loss: 38.0805 | NRF Acc: 0.1333
Epoch 02 | Joint Loss: 37.1997 | NRF Acc: 0.1381
Epoch 51 | Joint Loss: 3.8656 | NRF Acc: 0.3762
Epoch 101 | Joint Loss: 3.6196 | NRF Acc: 0.5452
Epoch 151 | Joint Loss: 3.2925 | NRF Acc: 0.5429
Epoch 201 | Joint Loss: 3.1310 | NRF Acc: 0.5976
Epoch 251 | Joint Loss: 2.9625 | NRF Acc: 0.6262
Epoch 301 | Joint Loss: 2.8047 | NRF Acc: 0.6476
Epoch 351 | Joint Loss: 2.6881 | NRF Acc: 0.6310
Epoch 401 | Joint Loss: 2.5634 | NRF Acc: 0.7143
Epoch 451 | Joint Loss: 2.4630 | NRF Acc: 0.6929
Epoch 501 | Joint Loss: 2.3900 | NRF Acc: 0.6905
Epoch 551 | Joint Loss: 2.3492 | NRF Acc: 0.6905
Epoch 601 | Joint Loss: 2.2968 | NRF Acc: 0.6976
Epoch 651 | Joint Loss: 2.2339 | NRF Acc: 0.6833
Epoch 701 | Joint Loss: 2.2008 | NRF Acc: 0.7262
Epoch 751 | Joint Loss: 2.1796 | NRF Acc: 0.7190
Epoch 801 | Joint Loss: 2.1387 | NRF Acc: 0.7452
Epoch 851 | Joint Loss: 2.1051 | NRF Acc: 0.7333
Epoch 901 | Joint Loss: 2.0852 | NRF Acc: 0.7310
Epoch 951 | Joint Los

In [ ]:
#best_acc, best_epoch, precision, recall, f1, y_pred_decoded, y_true_decoded, precision_micro, recall_micro, f1_micro,precision_macro, recall_macro, f1_macro,roc_auc_weighted, roc_auc_micro, roc_auc_macro,epoch_logs = train_joint_subgraph(traindata_list, testdata_list, y_gcn, y_nrf, nrf_input, default_args, index_to_label, verbose=True)

In [ ]:
"""
default_args = {
    'partition': f"gtd{partition}",
    'embed_dim': 16,
    'lr': 0.001,
    'epochs': 1500,
    'feat_dropout': 0,
    'n_tree': 80,
    'tree_depth': 10,
    'tree_feature_rate': 0.5,
    'n_class': 30,
    'batch_size': 256
}
"""
#Best acc/epoch: 0.8333 at epoch 1454


'\ndefault_args = {\n    \'partition\': f"gtd{partition}",\n    \'embed_dim\': 16,\n    \'lr\': 0.001,\n    \'epochs\': 1500,\n    \'feat_dropout\': 0,\n    \'n_tree\': 80,\n    \'tree_depth\': 10,\n    \'tree_feature_rate\': 0.5,\n    \'n_class\': 30,\n    \'batch_size\': 256\n}\n'

In [ ]:
#Best acc/epoch: 0.7856 at epoch 400
#Best acc/epoch: 0.8133 at epoch 1378


In [ ]:
best_acc

0.8355555534362793